# Image to 3D mesh on a free Colab T4 (Hunyuan3D-2mini)
Runtime -> Change runtime type -> **T4 GPU**, then run the cells in order. Upload your own picture in cell 3, or use the Hunyuan3D-2 demo picture.

In [ ]:
# 1. Install. --no-deps keeps Colab's own torch. Hunyuan3D's checkpoint uses transformers 4 layer names,
# so transformers is pinned below 5, and diffusers to a release that works with the matching huggingface-hub.
!pip -q install hy3dgen --no-deps
!pip -q install omegaconf einops trimesh pygltflib pyyaml safetensors opencv-python-headless rembg onnxruntime scikit-image pymeshlab fast-simplification
!pip -q install 'transformers>=4.49,<5' 'diffusers>=0.33,<0.36'

In [ ]:
# 2. Restart the Python process so numpy/scipy reload cleanly after the installs above.
# Colab will say the session crashed. That is expected. Continue with cell 3.
import os; os.kill(os.getpid(), 9)

## Model storage
The model is about 8 GB. It fits a free Google Drive, so it is saved there the first time and reused afterwards.
## Model storage: reuse it from Google Drive, download it only if it is missing
The next cell defines `get_model()`. It mounts Google Drive (Colab will ask you to allow it) and keeps models in `MyDrive/colab-models/`.
- **Already in Drive:** it is used as it is, with no download.
- **Missing and it fits:** it is downloaded straight into Drive, so the next session skips the download.
- **Missing and Drive is too small** (a free Drive account has only 15 GB in total): it downloads to this session's disk instead and tells you so.
- **Say no to the Drive prompt, or set `USE_DRIVE = False`:** it simply uses the session's disk.

In [ ]:
"""Model cache for Colab: reuse a model saved in Google Drive, download it if it is missing, fall back to the local disk if Drive is too small.

Paste this cell near the top of a notebook. Then:

    MODEL_DIR = get_model("black-forest-labs/FLUX.2-klein-4B", "flux2-klein-4b",
                          allow_patterns=[...], ignore_patterns=[...])

What it does, in order:
  1. Mounts Google Drive (skipped if USE_DRIVE = False or the mount is refused) and uses  MyDrive/colab-models/<name>.
  2. If that folder already holds a COMPLETE copy (a marker file written after a finished download), it is used as it is: no download.
  3. Otherwise it asks Hugging Face how big the download is and compares that with the free space in Drive:
       - it fits (with 5% spare): download straight into Drive, so the next session skips the download;
       - it does not fit: download to Colab's local disk (/content/models/<name>) and say so. Free Drive is only 15 GB in total.
  4. A download that was cut off is resumed, not restarted (huggingface_hub keeps partial files).
Nothing is deleted. Returns the folder to pass to from_pretrained().
"""
import os, shutil, json, time

USE_DRIVE = True                                     # set False to always use the local disk
DRIVE_ROOT = "/content/drive/MyDrive/colab-models"   # where models are kept in your Drive
LOCAL_ROOT = "/content/models"                       # used when Drive is off or too small
_MARK = ".download_complete.json"


def _mount_drive():
    if not USE_DRIVE:
        return False
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir("/content/drive/MyDrive")
    except Exception as e:                           # not in Colab, or the user said no
        print("Google Drive not available (" + type(e).__name__ + "): using the local disk")
        return False


def _remote_size(repo, allow_patterns, ignore_patterns):
    from huggingface_hub import HfApi
    import fnmatch
    total = 0
    for f in HfApi().list_repo_tree(repo, recursive=True):
        size = getattr(f, "size", None)
        if size is None:
            continue
        p = f.path
        if allow_patterns and not any(fnmatch.fnmatch(p, a) for a in allow_patterns):
            continue
        if ignore_patterns and any(fnmatch.fnmatch(p, i) for i in ignore_patterns):
            continue
        total += size
    return total


def get_model(repo, name, allow_patterns=None, ignore_patterns=None, revision=None):
    """Return a local folder holding `repo`, using Drive as a cache when it can."""
    from huggingface_hub import snapshot_download
    ignore_patterns = list(ignore_patterns or []) + [_MARK]
    drive_ok = _mount_drive()
    drive_dir = os.path.join(DRIVE_ROOT, name)
    local_dir = os.path.join(LOCAL_ROOT, name)

    for d, where in ((drive_dir, "Google Drive"), (local_dir, "this session's disk")):
        if os.path.exists(os.path.join(d, _MARK)):
            print(f"Using the saved copy in {where}: {d}  (no download)")
            return d

    need = _remote_size(repo, allow_patterns, ignore_patterns)
    print(f"{repo}: about {need / 1e9:.1f} GB to download")
    target = local_dir
    if drive_ok:
        os.makedirs(DRIVE_ROOT, exist_ok=True)
        used_by_us = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(drive_dir) for f in fs) if os.path.isdir(drive_dir) else 0
        free = shutil.disk_usage("/content/drive/MyDrive").free + used_by_us
        if need * 1.05 < free:
            target = drive_dir
            print(f"It fits in Google Drive ({free / 1e9:.1f} GB free): saving to {drive_dir} so the next session skips the download.")
        else:
            print(f"It does NOT fit in Google Drive ({free / 1e9:.1f} GB free, free accounts get 15 GB): using this session's disk, "
                  "so the next session downloads it again. Free up Drive space or use Google One to keep it.")
    os.makedirs(target, exist_ok=True)
    t = time.time()
    snapshot_download(repo, local_dir=target, allow_patterns=allow_patterns, ignore_patterns=ignore_patterns, revision=revision)
    json.dump({"repo": repo, "revision": revision, "bytes": need, "seconds": round(time.time() - t)}, open(os.path.join(target, _MARK), "w"))
    print(f"Done in {time.time() - t:.0f}s: {target}")
    return target

In [ ]:
HUNYUAN_DIR = get_model("tencent/Hunyuan3D-2mini", "hunyuan3d-2mini", allow_patterns=["hunyuan3d-dit-v2-mini-turbo/*"])
print("model files in", HUNYUAN_DIR)

In [ ]:
# Pick an input picture: upload one, or fall back to the demo picture from the Hunyuan3D-2 repo.
from google.colab import files
IMAGE = None
try:
    up = files.upload()
    IMAGE = next(iter(up), None)
except Exception:
    pass
if not IMAGE:
    IMAGE = 'input_demo.png'
    !wget -q https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2/main/assets/demo.png -O input_demo.png
print('Using', IMAGE)

In [ ]:
# 4. Generate the shape (turbo mini model, about 5 GB VRAM) and save a .glb
import time, torch
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    HUNYUAN_DIR, subfolder='hunyuan3d-dit-v2-mini-turbo',
    use_safetensors=True, device='cuda')
img = Image.open(IMAGE).convert('RGBA')
if img.getextrema()[3][0] == 255:  # no transparency, so cut out the background
    from hy3dgen.rembg import BackgroundRemover
    img = BackgroundRemover()(img)
t = time.time()
mesh = pipe(image=img, num_inference_steps=5, octree_resolution=320, num_chunks=20000,
            generator=torch.manual_seed(12345), output_type='trimesh')[0]
mesh = mesh.simplify_quadric_decimation(face_count=30000)  # the raw mesh is ~500k faces; 30k is plenty for a game
print(f'done in {time.time()-t:.1f}s, peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
mesh.export('model.glb')

In [ ]:
# 5. Download it. Open the .glb in Blender, Godot, three.js, or https://gltf-viewer.donmccurdy.com
from google.colab import files
files.download('model.glb')